In [1]:
import pandas as pd
import re
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Load dataset
df = pd.read_csv("histogram_simple.csv")

# Use first text column
text_col = df.select_dtypes(include="object").columns[0]

# NLP cleaning
stop = set(stopwords.words("english"))

def clean(x):
    x = re.sub(r"[^a-zA-Z\s]", "", str(x)).lower()
    return " ".join(w for w in x.split() if w not in stop)

df["clean"] = df[text_col].apply(clean)

# NLP -> TF-IDF
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(df["clean"])

# K-Means
k = 3
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = model.fit_predict(X)

# Show results
print(df[[text_col, "cluster"]])

# PCA visualization
pca = PCA(n_components=2)
points = pca.fit_transform(X.toarray())

plt.scatter(points[:, 0], points[:, 1], c=df["cluster"])
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means Clustering")
plt.show()

# Cluster histogram
df["cluster"].value_counts().sort_index().plot(
    kind="bar",
    title="Documents per Cluster"
)
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.show()

# Save results
df.to_csv("histogram_simple_clustered.csv", index=False)

IndexError: index out of bounds